# 10 · Объединённая цель: учитель-промпт, где он прав, иначе центроид верных состояний

**Зачем** (JOURNAL, прогон 09). Без градиента и без LoRA две цели аффинной поправки в точке вызова
дополняют друг друга: `linmap_prompt` (учитель — та же модель с промптом explicit_state) переносится на
новые типы требований (0.83 против 0.88 у LoRA), а `linmap` (центроид верных состояний базы) точнее на
знакомых (0.76 против 0.70). `linmap_mix` берёт цель учителя там, где он решил верно, иначе — цель `linmap`.

Те же 110 промптов train; выбор на val; сетка силы до 4.0 (у `linmap_prompt` 3.0 была на краю).
Проверка — на **свежем** наборе переноса (seed 404): набор 303 повлиял на выбор этого метода.
Результат **разведочный**.

In [2]:
%autoreload 0

In [3]:
cfg = ExperimentConfig.make(
    name="mix-target-qwen3-1.7b", hypothesis="explore", notebook="notebooks/10_mix_target.ipynb",
    notes="Объединённая цель linmap_mix: учитель-промпт, где он прав, иначе центроид верных состояний.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "lora", "explicit_state", "linmap", "linmap_prompt", "linmap_mix"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.ranks": [8, 16],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 404,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[16:11:32] run 20261031-161130_mix-target-qwen3-1.7b_9b4b1c  ->  results/runs/20261031-161130_mix-target-qwen3-1.7b_9b4b1c
[16:11:32] data: {'train': 79, 'val': 31, 'test': 50}  types={'invariant': 41, 'changing': 112, 'stress': 7}  positions={'early': 114, 'late': 46}


baseline/train:   0%|          | 0/158 [00:00<?, ?it/s]

[16:11:32] loading model Qwen/Qwen3-1.7B …


W1031 16:11:34.666000 17314 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (326 outputs left out)

linmap_prompt/val:   0%|          | 0/62 [00:00<?, ?it/s]

linmap_mix/val:   0%|          | 0/62 [00:00<?, ?it/s]

[16:11:49] val: baseline 0.621; lora 0.948; explicit_state 0.552; linmap 0.862; linmap_prompt 0.776; linmap_mix 0.810


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.621,0.621,0.526,0.4,NaN,,NaN,NaN,None,None,,0.50,831,10.0,400.6
1,lora,0.948,0.948,1.000,0.8,0.328,"[+0.190, +0.466]",19.0,0.0,None,None,,0.00,831,10.0,299.0
2,explicit_state,0.552,0.552,0.474,0.1,-0.069,"[-0.207, +0.069]",6.0,10.0,None,None,,0.25,1932,11.0,3443.7
3,linmap,0.862,0.862,0.842,0.6,0.241,"[+0.121, +0.379]",16.0,2.0,None,None,,0.50,831,10.0,410.6
4,linmap_prompt,0.776,0.776,0.684,0.6,0.155,"[+0.052, +0.276]",11.0,2.0,None,None,,0.50,831,10.0,410.9
5,linmap_mix,0.810,0.810,0.789,0.6,0.190,"[+0.086, +0.310]",11.0,0.0,None,None,,0.50,831,10.0,421.6


### Подбор на val

In [4]:
f = exp.state["fitted"]
t = f.get("linmap_mix", {})
print(f"учитель {t.get('teacher')} на train: успех {t.get('teacher_train_success', float('nan')):.3f} "
      f"(база {t.get('base_train_success', float('nan')):.3f})")
for m in ("linmap_mix", "linmap_prompt", "linmap"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = ["layer", "rank", "alpha", "train_r2", "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken"]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

учитель explicit_state на train: успех 0.736 (база 0.755)


**linmap_mix** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
27,20,16,2.0,0.721,0.810,0.737,0.6,-0.2,13,2
10,18,8,1.0,0.669,0.810,0.789,0.6,-0.2,11,0
15,18,16,1.0,0.734,0.810,0.789,0.6,-0.2,11,0
22,20,8,2.0,0.656,0.810,0.737,0.6,-0.2,12,1
23,20,8,3.0,0.656,0.810,0.579,0.8,-0.4,16,5
16,18,16,1.5,0.734,0.810,0.789,0.6,-0.2,11,0
6,16,16,1.5,0.774,0.793,0.737,0.6,-0.2,10,0
26,20,16,1.5,0.721,0.793,0.737,0.6,-0.2,11,1


**linmap_prompt** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
44,24,8,4.0,0.624,0.776,0.684,0.6,-0.2,11,2
54,26,8,4.0,0.611,0.759,0.632,0.6,-0.2,11,3
33,22,8,3.0,0.630,0.741,0.684,0.5,-0.1,8,1
23,20,8,3.0,0.668,0.741,0.632,0.6,-0.2,9,2
53,26,8,3.0,0.611,0.741,0.684,0.5,-0.1,9,2
43,24,8,3.0,0.624,0.741,0.684,0.5,-0.1,9,2
46,24,16,1.5,0.680,0.724,0.684,0.5,-0.1,7,1
34,22,8,4.0,0.630,0.724,0.632,0.6,-0.2,9,3


**linmap** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
16,18,16,1.5,0.568,0.862,0.789,0.8,-0.4,17,3
11,18,8,1.5,0.547,0.862,0.842,0.6,-0.2,16,2
26,20,16,1.5,0.561,0.828,0.737,0.6,-0.2,15,3
43,24,8,3.0,0.553,0.828,0.684,0.9,-0.5,18,6
17,18,16,2.0,0.568,0.828,0.684,0.7,-0.3,18,6
47,24,16,2.0,0.567,0.828,0.737,0.6,-0.2,15,3
15,18,16,1.0,0.568,0.810,0.842,0.5,-0.1,11,0
48,24,16,3.0,0.567,0.810,0.632,0.9,-0.5,18,7


### Свежий набор переноса (seed 404): знакомые и новые типы требований

In [5]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:  38%|███▊      | 38/100 [00:00<00:00, 374.16it/s]

baseline/transfer:seen:  77%|███████▋  | 77/100 [00:00<00:00, 379.61it/s]

lora/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

explicit_state/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

explicit_state/transfer:seen:  42%|████▏     | 42/100 [00:00<00:00, 418.92it/s]

... (30 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.60,0.60,0.167,0.643,NaN,,NaN,NaN,0.52,NaN,,None,838,10.0,445.3
1,lora,0.83,0.83,0.722,0.714,0.23,"[+0.110, +0.340]",30.0,7.0,0.73,0.21,"[+0.080, +0.330]",None,838,10.0,513.9
2,explicit_state,0.63,0.63,0.306,0.500,0.03,"[-0.060, +0.120]",17.0,14.0,NaN,NaN,,None,1982,11.0,5779.5
3,linmap,0.80,0.80,0.528,0.929,0.20,"[+0.120, +0.290]",23.0,3.0,0.67,0.15,"[+0.060, +0.250]",None,838,10.0,420.1
4,linmap_prompt,0.64,0.64,0.222,0.714,0.04,"[-0.050, +0.130]",18.0,14.0,0.62,0.10,"[+0.010, +0.190]",None,838,10.0,458.2
5,linmap_mix,0.81,0.81,0.528,0.929,0.21,"[+0.120, +0.300]",23.0,2.0,0.69,0.17,"[+0.090, +0.250]",None,838,10.0,428.1


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.37,0.37,0.029,0.400,NaN,,NaN,NaN,0.32,NaN,,None,860,10.0,406.9
1,lora,0.91,0.91,0.743,1.000,0.54,"[+0.440, +0.640]",54.0,0.0,0.84,0.52,"[+0.410, +0.630]",None,860,10.0,428.3
2,explicit_state,0.89,0.89,0.771,0.867,0.52,"[+0.410, +0.630]",52.0,0.0,NaN,NaN,,None,2024,11.0,5172.6
3,linmap,0.66,0.66,0.286,0.667,0.29,"[+0.190, +0.400]",29.0,0.0,0.50,0.18,"[+0.070, +0.300]",None,860,10.0,434.9
4,linmap_prompt,0.78,0.78,0.457,0.867,0.41,"[+0.300, +0.520]",41.0,0.0,0.67,0.35,"[+0.250, +0.450]",None,860,10.0,447.0
5,linmap_mix,0.68,0.68,0.314,0.667,0.31,"[+0.210, +0.420]",31.0,0.0,0.58,0.26,"[+0.160, +0.360]",None,860,10.0,441.2


### Цена: подготовка и вывод

In [6]:
d = exp.backend.hidden_size
cost = []
lr = f["lora"]["row"]
cost.append({"method": "lora", "prep": "градиент по весам (копия fp32)", "prep_seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m, how in (("linmap", "закрытая форма"), ("linmap_prompt", "закрытая форма + учитель-промпт (генерация)"),
               ("linmap_mix", "закрытая форма + учитель-промпт (генерация)")):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        cost.append({"method": m, "prep": how, "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": 2 * d * r["rank"] + d})
cost = pd.DataFrame(cost)
inf = exp.results_table("transfer:seen").set_index("method")[["tokens", "calls", "p50_ms"]]
cost.merge(inf, left_on="method", right_index=True, how="outer").reset_index(drop=True)

,method,prep,prep_seconds,params,tokens,calls,p50_ms
0,baseline,NaN,NaN,NaN,838,10.0,445.3
1,explicit_state,NaN,NaN,NaN,1982,11.0,5779.5
2,linmap,закрытая форма,108.2,3.482e+04,838,10.0,420.1
3,linmap_mix,закрытая форма + учитель-промпт (генерация),655.3,3.482e+04,838,10.0,428.1
4,linmap_prompt,закрытая форма + учитель-промпт (генерация),655.3,3.482e+04,838,10.0,458.2
5,lora,градиент по весам (копия fp32),671.9,3.211e+06,838,10.0,513.9


## Вывод → журнал

In [7]:
display(Markdown(exp.report(
    conclusion="""РАЗВЕДКА (hypothesis=explore, test не открыт). Объединённая цель linmap_mix (учитель-промпт там, где он прав, иначе центроид верных состояний базы), Qwen3-1.7B, те же 110 промптов train, отбор по val; перенос — свежий набор seed 404 (36/35 меняющих и 14/15 инвариантных пар).

1) linmap_mix (выбран L18, ранг 8, α=1.0) ведёт себя как linmap. Успех / чувств. / специф. (эпизоды):
- знакомые типы: база 0.60/0.17/0.64 (0.52); LoRA 0.83/0.72/0.71 (0.73); linmap_mix 0.81/0.53/0.93 (0.69); linmap 0.80/0.53/0.93 (0.67); linmap_prompt 0.64/0.22/0.71 (0.62); explicit_state 0.63/0.31/0.50.
- новые типы: база 0.37/0.03/0.40 (0.32); LoRA 0.91/0.74/1.00 (0.84); explicit_state 0.89/0.77/0.87; linmap_prompt 0.78/0.46/0.87 (0.67, +0.35 [+0.25, +0.45]); linmap_mix 0.68/0.31/0.67 (0.58); linmap 0.66/0.29/0.67 (0.50).
На знакомых типах linmap_mix на уровне LoRA по успеху (выше специфичность, ниже чувствительность), на новых — почти как linmap и заметно ниже linmap_prompt. Совместить сильные стороны двух целей не вышло: на выбранном слое L18 преобладает часть «центроид», а отбор по val снова взял конфигурацию, хорошую на знакомых типах (на val у linmap_mix шесть конфигураций с одинаковым успехом 0.810 на слоях 18 и 20; выбор между ними — по меньшей силе).

2) Повтор на третьем свежем наборе: linmap_prompt на новых типах 0.78 (в 09: 0.83) при базе 0.37 и LoRA 0.91 — перенос учителя-промпта подтверждается; на знакомых прироста нет (+0.04 [−0.05, +0.13]). linmap устойчив: знакомые 0.80, новые 0.66 (в 08: 0.85 / 0.61, в 09: 0.76 / 0.64).

3) У linmap_prompt лучший α=4.0 — снова край сетки: цели учителя нужны большие сдвиги.

4) Сам учитель explicit_state: новые 0.89, знакомые 0.63 (специф. 0.50); 5.2–5.8 с на решение против ~0.4 с у поправок. Цена подготовки та же: linmap 108 с, linmap_prompt и linmap_mix 655 с (генерация учителя), LoRA 672 с.

Ограничения: интервалы широкие; всё — разведка, подтверждать на отложенной выборке.""",
    next_step="""1) Ноутбук 11 (отбор val+loto, свежий набор seed 505) уже идёт: покажет, выберет ли отбор с отложенным типом у linmap_mix и linmap_prompt конфигурации, которые лучше переносятся. 2) Если linmap_mix и при новом отборе ≈ linmap — пробовать вес между целями (учитель / центроид) или их сумму вместо выбора по правоте учителя. 3) Расширить сетку α для linmap_prompt (4.0 на краю).""",
)))

## 2026-10-31 16:18 · прогон `20261031-161130_mix-target-qwen3-1.7b_9b4b1c`

**Гипотеза:** explore · **Ноутбук:** notebooks/10_mix_target.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, transfer · **git:** `675e567` (есть незакоммиченные изменения)

**Зачем:** Объединённая цель linmap_mix: учитель-промпт, где он прав, иначе центроид верных состояний.

**Отличия от конфига по умолчанию:** `act.alphas` = [1.0, 1.5, 2.0, 3.0, 4.0]; `act.ranks` = [8, 16]; `data.n_tasks` = 160; `episode.on_val` = False; `eval.methods` = ['baseline', 'lora', 'explicit_state', 'linmap', 'linmap_prompt', 'linmap_mix']; `model.name` = 'Qwen/Qwen3-1.7B'; `transfer.episode_methods` = ['baseline', 'lora', 'linmap', 'linmap_prompt', 'linmap_mix']; `transfer.seed` = 404

**⚠ Флаги протокола:**
- незакоммиченные изменения в src/docs/notebooks/configs — прогон не воспроизводится по коммиту

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.621 | 0.526 | 0.400 | — |  | — | — | 831 | 10.000 | 400.600 |
| lora | 0.948 | 1.000 | 0.800 | 0.328 | [+0.190, +0.466] | 19.000 | 0.000 | 831 | 10.000 | 299.000 |
| explicit_state | 0.552 | 0.474 | 0.100 | -0.069 | [-0.207, +0.069] | 6.000 | 10.000 | 1932 | 11.000 | 3443.700 |
| linmap | 0.862 | 0.842 | 0.600 | 0.241 | [+0.121, +0.379] | 16.000 | 2.000 | 831 | 10.000 | 410.600 |
| linmap_prompt | 0.776 | 0.684 | 0.600 | 0.155 | [+0.052, +0.276] | 11.000 | 2.000 | 831 | 10.000 | 410.900 |
| linmap_mix | 0.810 | 0.789 | 0.600 | 0.190 | [+0.086, +0.310] | 11.000 | 0.000 | 831 | 10.000 | 421.600 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.600 | 0.167 | 0.643 | — |  | — | — | 0.520 | — | 838 | 10.000 | 445.300 |
| lora | 0.830 | 0.722 | 0.714 | 0.230 | [+0.110, +0.340] | 30.000 | 7.000 | 0.730 | 0.210 | 838 | 10.000 | 513.900 |
| explicit_state | 0.630 | 0.306 | 0.500 | 0.030 | [-0.060, +0.120] | 17.000 | 14.000 | — | — | 1982 | 11.000 | 5779.500 |
| linmap | 0.800 | 0.528 | 0.929 | 0.200 | [+0.120, +0.290] | 23.000 | 3.000 | 0.670 | 0.150 | 838 | 10.000 | 420.100 |
| linmap_prompt | 0.640 | 0.222 | 0.714 | 0.040 | [-0.050, +0.130] | 18.000 | 14.000 | 0.620 | 0.100 | 838 | 10.000 | 458.200 |
| linmap_mix | 0.810 | 0.528 | 0.929 | 0.210 | [+0.120, +0.300] | 23.000 | 2.000 | 0.690 | 0.170 | 838 | 10.000 | 428.100 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.370 | 0.029 | 0.400 | — |  | — | — | 0.320 | — | 860 | 10.000 | 406.900 |
| lora | 0.910 | 0.743 | 1.000 | 0.540 | [+0.440, +0.640] | 54.000 | 0.000 | 0.840 | 0.520 | 860 | 10.000 | 428.300 |
| explicit_state | 0.890 | 0.771 | 0.867 | 0.520 | [+0.410, +0.630] | 52.000 | 0.000 | — | — | 2024 | 11.000 | 5172.600 |
| linmap | 0.660 | 0.286 | 0.667 | 0.290 | [+0.190, +0.400] | 29.000 | 0.000 | 0.500 | 0.180 | 860 | 10.000 | 434.900 |
| linmap_prompt | 0.780 | 0.457 | 0.867 | 0.410 | [+0.300, +0.520] | 41.000 | 0.000 | 0.670 | 0.350 | 860 | 10.000 | 447.000 |
| linmap_mix | 0.680 | 0.314 | 0.667 | 0.310 | [+0.210, +0.420] | 31.000 | 0.000 | 0.580 | 0.260 | 860 | 10.000 | 441.200 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L16/post_message: AUROC 0.980, acc 0.882 при перестановке 0.573
- H2: ✔ support — различает (H1), но ошибается в 47% меняющих пар: чувствительность 0.526, специфичность 0.400
- H3: ? inconclusive (val) — на val: успех 0.621 → 0.655 при L16/pre_action/α=1.0; решает только test
- H4: · not_run — нужны steering и LoRA, выбранные на val и оценённые на одних задачах
- ALT: · not_run — steering не сравнивался с промптом / пересказом / явным состоянием

**Вывод:** РАЗВЕДКА (hypothesis=explore, test не открыт). Объединённая цель linmap_mix (учитель-промпт там, где он прав, иначе центроид верных состояний базы), Qwen3-1.7B, те же 110 промптов train, отбор по val; перенос — свежий набор seed 404 (36/35 меняющих и 14/15 инвариантных пар).

1) linmap_mix (выбран L18, ранг 8, α=1.0) ведёт себя как linmap. Успех / чувств. / специф. (эпизоды):
- знакомые типы: база 0.60/0.17/0.64 (0.52); LoRA 0.83/0.72/0.71 (0.73); linmap_mix 0.81/0.53/0.93 (0.69); linmap 0.80/0.53/0.93 (0.67); linmap_prompt 0.64/0.22/0.71 (0.62); explicit_state 0.63/0.31/0.50.
- новые типы: база 0.37/0.03/0.40 (0.32); LoRA 0.91/0.74/1.00 (0.84); explicit_state 0.89/0.77/0.87; linmap_prompt 0.78/0.46/0.87 (0.67, +0.35 [+0.25, +0.45]); linmap_mix 0.68/0.31/0.67 (0.58); linmap 0.66/0.29/0.67 (0.50).
На знакомых типах linmap_mix на уровне LoRA по успеху (выше специфичность, ниже чувствительность), на новых — почти как linmap и заметно ниже linmap_prompt. Совместить сильные стороны двух целей не вышло: на выбранном слое L18 преобладает часть «центроид», а отбор по val снова взял конфигурацию, хорошую на знакомых типах (на val у linmap_mix шесть конфигураций с одинаковым успехом 0.810 на слоях 18 и 20; выбор между ними — по меньшей силе).

2) Повтор на третьем свежем наборе: linmap_prompt на новых типах 0.78 (в 09: 0.83) при базе 0.37 и LoRA 0.91 — перенос учителя-промпта подтверждается; на знакомых прироста нет (+0.04 [−0.05, +0.13]). linmap устойчив: знакомые 0.80, новые 0.66 (в 08: 0.85 / 0.61, в 09: 0.76 / 0.64).

3) У linmap_prompt лучший α=4.0 — снова край сетки: цели учителя нужны большие сдвиги.

4) Сам учитель explicit_state: новые 0.89, знакомые 0.63 (специф. 0.50); 5.2–5.8 с на решение против ~0.4 с у поправок. Цена подготовки та же: linmap 108 с, linmap_prompt и linmap_mix 655 с (генерация учителя), LoRA 672 с.

Ограничения: интервалы широкие; всё — разведка, подтверждать на отложенной выборке.

**Следующий шаг:** 1) Ноутбук 11 (отбор val+loto, свежий набор seed 505) уже идёт: покажет, выберет ли отбор с отложенным типом у linmap_mix и linmap_prompt конфигурации, которые лучше переносятся. 2) Если linmap_mix и при новом отборе ≈ linmap — пробовать вес между целями (учитель / центроид) или их сумму вместо выбора по правоте учителя. 3) Расширить сетку α для linmap_prompt (4.0 на краю).
